# ICT-4202 | Artificial Intelligence and Machine Learning Laboratory
## Final Laboratory Examination — Set 10
**Islamic University, Kushtia, Bangladesh**  
Department of Information and Communication Technology

> **Notebook purpose:** Question-matched solutions with brief background theory, mathematical notation, readable Python, reproducible output, interpretation, and viva revision. This is a learning/reporting notebook, not a claim of official model answers.

| Question | Topic | Marks |
|:--|:--|--:|
| Q1 | Integrated forward-chaining agent | 25 |
| Q2 | BFS, Greedy, A* with assigned costs | 35 |
| Q3 | Iris classification metrics comparison | 30 |
| Q4 | Viva: knowledge base, shortest versus cheapest, F1 | 10 |

**Execution:** Open in JupyterLab/VS Code, choose a Python kernel, then use **Run All**. All random experiments use a fixed seed. Markdown cells use MathJax (`$...$` and `$$...$$`) to render equations on GitHub/Jupyter.

**Quick access - one file per numbered question:** [Q1 / A](Set_10_A.py) | [Q2 / B](Set_10_B.py) | [Q3 / C](Set_10_C.py) | [Q4 / D: Viva practice](Set_10_D.py).

Q1-Q3 contain the required practical work (including all subparts). Q4 is a viva, so its `.py` file is a revision/self-quiz tool, not an extra algorithm assignment. This notebook contains solutions, explanations, mathematical theory, and the Q4 viva questions.

---
## Q1. Integrated Knowledge-Based Agent (25 marks)
**Question specification**  
`TemperatureHigh`; `TemperatureHigh→Hot`; `Hot AND Humid→Uncomfortable`; `Uncomfortable→FanOn`; `Rain→Humid`. Run forward chaining for: high temperature only; high temperature + Humid; high temperature + Rain; show traces and given-versus-inferred facts.

### Theory and mathematical background
A knowledge base (KB) stores **facts** and **rules**. In propositional forward chaining, rules have premises and a conclusion. For example:
$$P\land Q\Rightarrow R.$$
A rule fires when all its premises are already known. This is repeated until no new conclusion can be inferred (the *fixed point*).

For implication, $P\Rightarrow Q\equiv \neg P\lor Q$; it is false only when $P$ is true and $Q$ is false. Forward chaining for definite Horn-style rules is a data-driven form of logical inference.

In [1]:

def forward_chain(initial_facts, rules):
    """Data-driven inference: rules are (premise-set, conclusion) pairs."""
    known = set(initial_facts)
    trace = []
    while True:
        newly_added = False
        for premises, conclusion in rules:
            if premises.issubset(known) and conclusion not in known:
                known.add(conclusion)
                trace.append((sorted(premises), conclusion))
                newly_added = True
        if not newly_added:
            break
    return known, trace

def print_inference(label, initial, rules):
    known, trace = forward_chain(initial, rules)
    print(f"\n{label} | Initial facts: {sorted(initial)}")
    for i, (prem, conclusion) in enumerate(trace, 1):
        print(f"  {i}. {prem}  =>  {conclusion}")
    if not trace: print("  No new rule fired.")
    print('Final facts:', sorted(known))
    return known, trace

rules=[({'TemperatureHigh'},'Hot'),({'Hot','Humid'},'Uncomfortable'),
       ({'Uncomfortable'},'FanOn'),({'Rain'},'Humid')]
cases=[('TemperatureHigh only',{'TemperatureHigh'}),
       ('TemperatureHigh + Humid',{'TemperatureHigh','Humid'}),
       ('TemperatureHigh + Rain',{'TemperatureHigh','Rain'})]
for title,given in cases:
    known,trace=print_inference(title,given,rules)
    print('Given (asserted):',sorted(given))
    print('Derived (inferred):',sorted(known-given),'| FanOn?', 'FanOn' in known)



TemperatureHigh only | Initial facts: ['TemperatureHigh']
  1. ['TemperatureHigh']  =>  Hot
Final facts: ['Hot', 'TemperatureHigh']
Given (asserted): ['TemperatureHigh']
Derived (inferred): ['Hot'] | FanOn? False

TemperatureHigh + Humid | Initial facts: ['Humid', 'TemperatureHigh']
  1. ['TemperatureHigh']  =>  Hot
  2. ['Hot', 'Humid']  =>  Uncomfortable
  3. ['Uncomfortable']  =>  FanOn
Final facts: ['FanOn', 'Hot', 'Humid', 'TemperatureHigh', 'Uncomfortable']
Given (asserted): ['Humid', 'TemperatureHigh']
Derived (inferred): ['FanOn', 'Hot', 'Uncomfortable'] | FanOn? True

TemperatureHigh + Rain | Initial facts: ['Rain', 'TemperatureHigh']
  1. ['TemperatureHigh']  =>  Hot
  2. ['Rain']  =>  Humid
  3. ['Hot', 'Humid']  =>  Uncomfortable
  4. ['Uncomfortable']  =>  FanOn
Final facts: ['FanOn', 'Hot', 'Humid', 'Rain', 'TemperatureHigh', 'Uncomfortable']
Given (asserted): ['Rain', 'TemperatureHigh']
Derived (inferred): ['FanOn', 'Hot', 'Humid', 'Uncomfortable'] | FanOn? True


### Result interpretation and discussion
In the first case, `Hot` is inferred but `Humid` is missing. In the second case, Humid is **given** and `Uncomfortable`/`FanOn` follow. In the third, `Rain` is **given** and `Humid` is **inferred** before the joint rule can fire. The final conclusions match the same logical implication chain.

---
## Q2. Integrated Search Task (35 marks)
**Question specification**  
`S→{A,B}, A→{C,D}, B→{D,E}, C→{G}, D→{G}, E→{G}`. Assign *positive costs* and a *reasonable heuristic* (not specified by question); run BFS, Greedy and A*, showing expansion traces, route and cost. Explain minimum edges versus minimum cost.

### Theory and mathematical background
For an unweighted graph, **BFS** expands the shallowest frontier state first (FIFO queue), so it finds a minimum-edge path when each edge has equal cost. **DFS** explores a branch deeply first (LIFO stack), so its outcome depends on neighbor order. A **visited set** prevents re-expansion.

With $V$ vertices and $E$ edges, BFS and DFS can both run in $O(V+E)$ time on an adjacency-list graph; their frontier sizes and discovered-node storage differ. In tree-search notation, BFS typically uses $O(b^{d+1})$ space, while DFS uses $O(bm)$ space ($b$ branching factor, $d$ shallowest goal depth, $m$ maximum depth). These are different models of complexity.

A heuristic $h(n)$ estimates remaining cost from state $n$ to the goal. **Greedy best-first search** prioritizes only $h(n)$, so it need not find the lowest-cost route.

**A\* search** uses
$$g(n)=\text{cost from start to }n,\qquad f(n)=g(n)+h(n).$$
A heuristic is **admissible** when
$$0\le h(n)\le h^*(n),$$
where $h^*(n)$ is the true cheapest remaining cost. Under suitable graph-search conditions, admissible heuristics with cost updates/reopening allow optimal A\*; non-admissible heuristics remove this guarantee.

In [2]:
from collections import deque

import heapq

from itertools import count

def bfs(graph, start, goal):
    queue = deque([(start, [start])])
    seen = {start}
    expanded = []
    while queue:
        node, path = queue.popleft()
        expanded.append(node)
        if node == goal:
            return path, expanded
        for nxt in graph.get(node, []):
            if nxt not in seen:
                seen.add(nxt)
                queue.append((nxt, path + [nxt]))
    return None, expanded

from itertools import count

import heapq

def astar(graph, h, start, goal):
    counter = count()
    heap = [(h[start], next(counter), 0, start, [start])]
    best_g = {start: 0}
    expanded = []
    while heap:
        f, _, g, node, path = heapq.heappop(heap)
        if g > best_g.get(node, float('inf')): continue  # stale entry
        expanded.append((node, g, h[node], f))
        if node == goal: return path, g, expanded
        for nxt, weight in graph.get(node, []):
            new_g = g + weight
            if new_g < best_g.get(nxt, float('inf')):
                best_g[nxt] = new_g
                heapq.heappush(heap, (new_g+h[nxt], next(counter), new_g, nxt, path+[nxt]))
    return None, float('inf'), expanded

def greedy_weighted(graph, h, start, goal):
    counter = count()
    heap = [(h[start], next(counter), 0, start, [start])]
    seen, expanded = set(), []
    while heap:
        _, _, g, node, path = heapq.heappop(heap)
        if node in seen: continue
        seen.add(node)
        expanded.append((node, g, h[node]))
        if node == goal: return path, g, expanded
        for nxt, weight in graph.get(node, []):
            if nxt not in seen:
                heapq.heappush(heap, (h[nxt], next(counter), g+weight, nxt, path+[nxt]))
    return None, float('inf'), expanded

def display_astar_trace(trace):
    print("Node     g(n)     h(n)     f(n)")
    for node, g, h, f in trace:
        print(f"{node:>4} {g:>8} {h:>8} {f:>8}")

# Positive weights and an admissible (but deliberately uneven) heuristic are
# our EXPLICIT CHOICES: the examination question does not specify numbers.
weighted={'S':[('A',1),('B',2)],'A':[('C',9),('D',8)],
          'B':[('D',1),('E',4)],'C':[('G',7)],'D':[('G',2)],
          'E':[('G',6)],'G':[]}
unweighted={u:[v for v,c in edges] for u,edges in weighted.items()}
h={'S':4,'A':1,'B':3,'C':2,'D':1,'E':5,'G':0}
edge_cost={(u,v):cost for u,edges in weighted.items() for v,cost in edges}
def path_cost(path):
    return sum(edge_cost[(u,v)] for u,v in zip(path,path[1:])) if path else None
bfs_path,bfs_trace=bfs(unweighted,'S','G')
greedy_path,greedy_cost,greedy_trace=greedy_weighted(weighted,h,'S','G')
a_path,a_cost,a_trace=astar(weighted,h,'S','G')
print('BFS    :',bfs_path,'| expanded:',bfs_trace,'| cost:',path_cost(bfs_path))
print('Greedy :',greedy_path,'| expanded:',greedy_trace,'| cost:',greedy_cost)
print('A*     :',a_path,'| expanded:',a_trace,'| cost:',a_cost)
display_astar_trace(a_trace)
assert a_cost == 5 and path_cost(bfs_path) > a_cost


BFS    : ['S', 'A', 'C', 'G'] | expanded: ['S', 'A', 'B', 'C', 'D', 'E', 'G'] | cost: 17
Greedy : ['S', 'A', 'D', 'G'] | expanded: [('S', 0, 4), ('A', 1, 1), ('D', 9, 1), ('G', 11, 0)] | cost: 11
A*     : ['S', 'B', 'D', 'G'] | expanded: [('S', 0, 4, 4), ('A', 1, 1, 2), ('B', 2, 3, 5), ('D', 3, 1, 4), ('G', 5, 0, 5)] | cost: 5
Node     g(n)     h(n)     f(n)
   S        0        4        4
   A        1        1        2
   B        2        3        5
   D        3        1        4
   G        5        0        5


### Result interpretation and discussion
**Why the two criteria differ:** all four S→G routes in the given topology contain *three edges*. BFS therefore chooses one of several **minimum-edge** routes using adjacency order, here `S→A→C→G` with assigned cost $1+9+7=17$. A* evaluates weights and chooses `S→B→D→G` with lower cost $2+1+2=5$. Greedy may be misled by the smaller $h(A)$ and select a more expensive route. These cost numbers and heuristics are **illustrative author choices**, not values printed in the exam sheet.

---
## Q3. Classification (30 marks)
**Question specification**  
Using Iris 70/30 stratified split, compare k-NN $k=3$ and decision tree `max_depth=3`, with accuracy, confusion matrices, precision/recall/F1. Compare errors and state which model you prefer for this fixed run.

### Theory and mathematical background
**k-Nearest Neighbors (k-NN)** predicts the majority label among the $k$ nearest training samples. A common distance is Euclidean:
$$d(\mathbf x,\mathbf z)=\sqrt{\sum_{j=1}^{p}(x_j-z_j)^2}.$$
Standardization rescales a feature using training-set statistics:
$$z_j=\frac{x_j-\mu_j^{\mathrm{train}}}{\sigma_j^{\mathrm{train}}}.$$
**Accuracy** $=(TP+TN)/(TP+TN+FP+FN)$ for binary labels (or total correct / total cases for multiclass). A confusion matrix $C_{ij}$ counts instances of actual class $i$ predicted as class $j$. Fit scaling on training samples only to prevent leakage.

A **decision tree** recursively chooses feature thresholds that reduce node impurity. One measure is Gini impurity:
$$Gini(D)=1-\sum_{c=1}^{K}p(c\mid D)^2.$$
Restricting `max_depth` reduces complexity but can cause underfitting; large or unrestricted depth can memorize training samples and overfit.

For each class, $Precision=\frac{TP}{TP+FP}$, $Recall=\frac{TP}{TP+FN}$, and $F_1=\frac{2PR}{P+R}$. Macro averages assign equal weight to every class.

In [3]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import pandas as pd
import numpy as np
iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30,
                                                    stratify=y, random_state=42)
print('Iris samples:', X.shape, '| train:', len(y_train), '| test:', len(y_test))
print('Class order:', list(iris.target_names))

from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_recall_fscore_support
candidates={'k-NN (k=3)':KNeighborsClassifier(n_neighbors=3),
            'Decision tree (depth=3)':DecisionTreeClassifier(max_depth=3,random_state=42)}
rows=[]
for name,model in candidates.items():
    model.fit(X_train,y_train)
    pred=model.predict(X_test)
    accuracy=accuracy_score(y_test,pred)
    p,r,f1,_=precision_recall_fscore_support(y_test,pred,average='macro',zero_division=0)
    rows.append({'model':name,'accuracy':accuracy,'precision_macro':p,
                 'recall_macro':r,'f1_macro':f1})
    errors=np.flatnonzero(pred!=y_test)
    print('\n'+name)
    print('Confusion matrix (actual rows):\n',confusion_matrix(y_test,pred))
    print('Misclassified TEST ROW POSITIONS:',errors.tolist())
    for idx in errors[:10]:
        print('  Row',idx,'| actual:',iris.target_names[y_test[idx]],
              '| predicted:',iris.target_names[pred[idx]])
comparison=pd.DataFrame(rows)
print(comparison.round(4).to_string(index=False))
winner=comparison.sort_values(['accuracy','f1_macro','model'],ascending=[False,False,True]).iloc[0]
print('Preferred for this particular split (accuracy then macro F1):',winner['model'])


Iris samples: (150, 4) | train: 105 | test: 45
Class order: [np.str_('setosa'), np.str_('versicolor'), np.str_('virginica')]

k-NN (k=3)
Confusion matrix (actual rows):
 [[15  0  0]
 [ 0 15  0]
 [ 0  2 13]]
Misclassified TEST ROW POSITIONS: [2, 42]
  Row 2 | actual: virginica | predicted: versicolor
  Row 42 | actual: virginica | predicted: versicolor

Decision tree (depth=3)
Confusion matrix (actual rows):
 [[15  0  0]
 [ 0 14  1]
 [ 0  0 15]]
Misclassified TEST ROW POSITIONS: [39]
  Row 39 | actual: versicolor | predicted: virginica
                  model  accuracy  precision_macro  recall_macro  f1_macro
             k-NN (k=3)    0.9556           0.9608        0.9556    0.9554
Decision tree (depth=3)    0.9778           0.9792        0.9778    0.9778
Preferred for this particular split (accuracy then macro F1): Decision tree (depth=3)


### Result interpretation and discussion
The preferred algorithm is chosen using the **displayed metrics for this fixed split**, prioritizing accuracy and breaking ties by macro F1. Another split may reverse the ranking. The 45-case test set is relatively small, and neither method is automatically superior; real research would compare repeated cross-validation and independent holdout performance.

---
## Q4. Viva voce (10 marks)

| Question | Concise answer |
|:--|:--|
| What is an asserted fact? | A proposition initially placed into the KB. |
| What is an inferred fact? | A new proposition derived using rules. |
| Difference between BFS and A*? | BFS minimizes edge count; A* uses costs g+h. |
| Greedy drawback? | It may favor a low h while ignoring high path cost g. |
| Confusion matrix? | Counts of actual versus predicted classes. |
| Precision? | Share of predicted positives that are truly positive. |
| Recall? | Share of actual positives correctly predicted. |
| F1? | Harmonic mean of precision and recall. |

| What is forward chaining? | Start with known facts and repeatedly apply rules whose premises are satisfied. |
| Does a shortest path in edges always have the lowest cost? | No. When edges have different costs, a path with fewer edges can cost more. |
| What is the F1 score? | The harmonic mean of precision and recall: F1 = 2PR/(P+R), when P+R is positive. |

**Review tip:** Explain the code by identifying the inputs, data structures, stopping conditions, evaluation metrics, and one limitation for each task.

---
### Submission checklist
- [ ] Execute every Python cell in sequence; check the saved output is current.
- [ ] Explain the main formula, trace, and interpretation for Q1, Q2, Q3.
- [ ] Record the original task settings, dataset, random seed, and actual test metrics.
- [ ] For WEKA tasks, run WEKA locally and attach the real result-buffer output/screenshot; notebook instructions **do not substitute for WEKA evidence**.
- [ ] Be ready to discuss one methodological limitation in viva.

**Integrity note:** Code and explanations are independently written for this study repository. Numeric results are calculated from the shown code and may vary with library versions where numerical or tie-breaking differences occur.